In [20]:
from langchain_ollama import ChatOllama
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.tools import Tool
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from sympy import sympify
import re

In [21]:
class AgentState(dict):
    input: str
    chat_history: list
    output: str
    context: str = ""
    tool_choice: str = ""
    done: bool = False

In [22]:
def safe_calculator(expression: str) -> str:
    """Calculadora segura usando sympy."""
    try:
        result = sympify(expression).evalf()
        return str(result)
    except Exception as e:
        return f"Error: {e}"

calculator_tool = Tool(
    name="CalculadoraSegura",
    func=safe_calculator,
    description="Resuelve operaciones matemáticas básicas (usa +, -, *, /, **, sqrt(x), etc.)"
)

wiki = WikipediaAPIWrapper(lang="es", top_k_results=2, doc_content_chars_max=800)
wiki_tool = Tool(
    name="BuscadorWikipedia",
    func=wiki.run,
    description="Busca información en Wikipedia en español sobre personas, fechas o eventos históricos."
)

TOOLS = {
    "CalculadoraSegura": calculator_tool,
    "BuscadorWikipedia": wiki_tool,
}

In [23]:
llm = ChatOllama(model="mistral")

In [24]:
def llm_router(state: AgentState):
    """El modelo analiza la pregunta y decide qué herramienta usar."""
    prompt = f"""
Eres un agente inteligente con acceso a estas herramientas:

1. BuscadorWikipedia - {wiki_tool.description}
2. CalculadoraSegura - {calculator_tool.description}

Decide cuál herramienta usar para resolver la siguiente pregunta.
Responde únicamente en este formato JSON:

{{
  "tool": "<nombre de la herramienta>",
  "input": "<texto o expresión para pasar a la herramienta>"
}}

Ejemplo:
Usuario: ¿Quién fue Marie Curie?
Respuesta:
{{"tool": "BuscadorWikipedia", "input": "Marie Curie"}}

Pregunta del usuario:
{state['input']}
"""

    response = llm.invoke(prompt)
    text = response.content.strip()

    # Intentamos extraer el JSON de la salida del modelo
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        json_text = match.group()
        try:
            import json
            data = json.loads(json_text)
            tool_name = data.get("tool", "")
            tool_input = data.get("input", "")
        except Exception:
            tool_name, tool_input = "", ""
    else:
        tool_name, tool_input = "", ""

    # Guardamos la elección
    state["tool_choice"] = tool_name
    state["tool_input"] = tool_input
    state["done"] = False
    return state

In [25]:
def execute_tool(state: AgentState):
    """Ejecuta la herramienta elegida por el modelo."""
    tool_name = state.get("tool_choice")
    tool_input = state.get("tool_input", "")

    if not tool_name or tool_name not in TOOLS:
        state["context"] = "No se identificó una herramienta adecuada."
        return state

    tool = TOOLS[tool_name]
    try:
        result = tool.func(tool_input)
        state["context"] = f"Resultado de {tool_name}: {result}"
    except Exception as e:
        state["context"] = f"Error ejecutando {tool_name}: {e}"

    state["done"] = False
    return state

In [26]:
def reasoner(state: AgentState):
    """El LLM usa la memoria + contexto de herramientas para generar la respuesta final."""
    context = state.get("context", "")
    prompt = f"""
Eres un asistente que razona usando información previa.
Información recuperada:
{context}

Historial previo:
{state.get('chat_history', [])}

Pregunta del usuario:
{state['input']}

Usa los datos recuperados para dar una respuesta breve y clara.
Si hay datos numéricos, razona con ellos.
"""
    response = llm.invoke(prompt)
    state["output"] = response.content
    state["done"] = True
    return state


In [27]:
graph = StateGraph(AgentState)

graph.add_node("router", llm_router)
graph.add_node("execute_tool", execute_tool)
graph.add_node("reasoner", reasoner)

graph.set_entry_point("router")
graph.add_edge("router", "execute_tool")
graph.add_edge("execute_tool", "reasoner")
graph.add_edge("reasoner", END)

# ===============================
# 💾 MEMORIA
# ===============================
memory = MemorySaver()
app = graph.compile(checkpointer=memory)


In [28]:
if __name__ == "__main__":
    config = {"configurable": {"thread_id": "alejandro-llm-router"}}

    print("\n💬 Pregunta 1:")
    result1 = app.invoke(
        {"input": "¿En qué año nació Marie Curie?", "chat_history": []},
        config=config
    )
    print(result1["output"])

    print("\n💬 Pregunta 2:")
    result2 = app.invoke(
        {"input": "¿Cuántos años tenía cuando ganó su primer Nobel?"},
        config=config
    )
    print(result2["output"])

    print("\n💬 Pregunta 3:")
    result3 = app.invoke(
        {"input": "Calcula la raíz cuadrada de 625 + 13*2"},
        config=config
    )
    print(result3["output"])


💬 Pregunta 1:
 Para encontrar el año de nacimiento de Marie Curie, no se necesita el BuscadorWikipedia que está dando error actualmente. Puedo informarte que Marie Curie nació en 1867.

💬 Pregunta 2:
 Debido a que no se proporciona información sobre un individuo en específico para responder su pregunta, no puedo dar una respuesta precisa sobre cuántos años tenía cuando ganó su primer Nobel. Por favor, verifique la documentación del BuscadorWikipedia y corrija el error reportado antes de intentar nuevamente.

💬 Pregunta 3:
 El resultado de tu operación es incorrecto debido a un error sintáctico en la expresión. La correcta sería: `(625 + 13*2)^(1/2)`. Calculando esto nos da como resultado aproximadamente `17.18`.
